In [ ]:
%load_ext autoreload
%autoreload 2

from itertools import product
import sys
sys.path.append('/home/projects/nyosef/zvise/PixelGen/')
from PixelGen.multimodalvi import MultiModalSCVI
from PixelGen.multimodalvae import MultiModalVAE, AggMethod, D
from PixelGen.enums import AggMethod, D
from PixelGen.metrics import MultiModalVIMetrics
from sklearn.preprocessing import PowerTransformer
from pathlib import Path

import anndata as ad
import torch
import scvi
import scipy
# from scvi import autotune

import seaborn as sns
import scanpy as sc
import pandas as pd
import numpy as np
from matplotlib import pyplot as pltpyg
from tqdm import tqdm

from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

# import ray
# from ray import tune


from PixelGen.pxl_utils import train_model, get_model_latents
from PixelGen.scvi_utils import plot_losses, pca_neighbors_umap, calc_PCA
from pixelator.common.statistics import clr_transformation, dsb_normalize


from pixelator.pna.plot import molecule_rank_plot
# from pixelator.plot import molecule_rank_plot, cell_count_plot, scatter_umi_per_upia_vs_tau
# from pixelator.statistics import c
# lr_transformation
# from pixelator.analysis.normalization import dsb_normalize


import tempfile

from torch.distributions import NegativeBinomial, Normal, Poisson, MixtureSameFamily, Beta
from torch.distributions import kl_divergence as kl

print(torch.cuda.is_available())
from sklearn.decomposition import PCA

from anndata import AnnData
scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)
sc.set_figure_params(figsize=(6, 6), frameon=False)

sns.set_theme()
torch.set_float32_matmul_precision("high")
save_dir = tempfile.TemporaryDirectory()

from utils import plot_latent, plot_gene_heatmap, plot_model_latents
from doublet_seperation import B_CD4_logfc_dict, B_CD8_logfc_dict, run_cellwise_coloc_analysis_to_disk, concat_abundance_to_adata, add_doublets_metadata
%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"
import pickle
import scipy.sparse as sp
from scipy.sparse.csgraph import dijkstra
import anndata
import hotspot

import numpy as np
import pandas as pd
import networkx as nx
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import pandas as pd
import torch
import os
from torch_geometric.data import Data
from tqdm import tqdm  # Progress bar
import glob
from pixelator.pna import read

In [ ]:
DATA_DIR = Path("/home/projects/nyosef/zvise/PxlgnProject/Data")
ANNOTATED_ADATA_PATH='/home/projects/nyosef/zvise/PixelGen/PixelGen/Data/adatas/final_adatas/adata_annotated.h5ad'

files = [f for f in DATA_DIR.rglob('*.pxl') if f.is_file()]
data = read(files)
adata=sc.read_h5ad(ANNOTATED_ADATA_PATH)

In [ ]:
output_dir = "/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/GVAE/graphs"
files = glob.glob(f"{output_dir}/*.pt")
ALL_MARKERS = adata.var_names.tolist()
MARKER_TO_IDX = {m: i for i, m in enumerate(ALL_MARKERS)}

# CREATE GRAPHS

In [ ]:
# --- SETUP ---
output_dir = "/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/GVAE/graphs"
os.makedirs(output_dir, exist_ok=True)

# Define Vocabulary from your AnnData object
# This ensures every graph has the exact same feature columns (159 proteins)
ALL_MARKERS = adata.var_names.tolist()
MARKER_TO_IDX = {m: i for i, m in enumerate(ALL_MARKERS)}

# --- PART 1: CONVERTER FUNCTION ---
def df_to_pyg_data(edge_df: pd.DataFrame) -> Data:
    """
    Converts edge DataFrame -> PyTorch Geometric Data object.
    """
    # 1. Map UMI strings to Integer IDs
    # Stack columns to find all unique nodes
    all_umis = pd.concat([edge_df["umi1"], edge_df["umi2"]])
    node_codes, unique_umis = pd.factorize(all_umis)

    # 2. Create Edge Index (Connectivity)
    src = node_codes[:len(edge_df)]
    dst = node_codes[len(edge_df):]
    edge_index = torch.tensor([src, dst], dtype=torch.long)

    # 3. Create Node Features (Protein Identity)
    # Lookup which protein corresponds to which UMI
    # We create a temporary lookup table from the dataframe itself
    node_lookup = pd.concat([
        edge_df[["umi1", "marker_1"]].rename(columns={"umi1": "umi", "marker_1": "marker"}),
        edge_df[["umi2", "marker_2"]].rename(columns={"umi2": "umi", "marker_2": "marker"})
    ]).drop_duplicates(subset="umi").set_index("umi")

    # Reorder markers to match the integer IDs from step 1
    ordered_markers = node_lookup.loc[unique_umis]["marker"]

    # Map string markers to integers [0, 158]
    x_indices = ordered_markers.map(MARKER_TO_IDX).fillna(0).values.astype(int)

    # One-Hot Encode: Shape [Num_Nodes, 159]
    x = torch.nn.functional.one_hot(
        torch.from_numpy(x_indices).long(), 
        num_classes=len(ALL_MARKERS)
    ).float()

    return Data(x=x, edge_index=edge_index)

# --- PART 2: THE SAVING LOOP ---
cells = adata.obs_names.tolist()

print(f"Processing {len(cells)} cells...")

for cell in tqdm(cells):
    # 1. Build the edge DataFrame
    # Note: 'data' refers to your pixelgen/spatial object
    edge_df = data.filter(components=cell).edgelist().to_df()
    
    # Skip empty edge lists
    if edge_df.empty:
        continue
        
    # 2. Convert to PyG Tensor
    try:
        pyg_data = df_to_pyg_data(edge_df)
        
        # 3. Save to disk
        torch.save(pyg_data, os.path.join(output_dir, f"{cell}.pt"))
        
    except Exception as e:
        print(f"Skipping {cell} due to error: {e}")

print("Done! All graphs saved.")

In [ ]:
import torch
import glob
import random
import os

# 1. Get list of all saved files
files = glob.glob(f"{output_dir}/*.pt")

if not files:
    print("❌ No files found in 'processed_cells/'. Check your saving loop.")
else:
    print(f"✅ Found {len(files)} files.")
    
    # 2. Inspect 3 random files
    print("\n--- Inspecting 3 Random Samples ---")
    for i in range(3):
        f = random.choice(files)
        data = torch.load(f)
        
        print(f"\n📂 File: {os.path.basename(f)}")
        print(f"   • Nodes: {data.num_nodes}")
        print(f"   • Edges: {data.num_edges}")
        print(f"   • Feature Matrix (x): {data.x.shape}  <-- Should be [Nodes, 159]")
        print(f"   • Connectivity (edge_index): {data.edge_index.shape} <-- Should be [2, Edges]")
        
        # 3. Sanity Checks
        # Check if features are actually One-Hot (Sum should be 1.0 per node)
        is_one_hot = torch.allclose(data.x.sum(dim=1), torch.tensor(1.0))
        print(f"   • Is One-Hot Valid? {is_one_hot}")
        
        # Check max edge index matches num_nodes
        if data.num_edges > 0:
            max_idx = data.edge_index.max().item()
            print(f"   • Max Node ID in edges: {max_idx} (Should be < {data.num_nodes})")

        # Peek at the first node's feature
        # finding which protein is active for node 0
        active_protein_idx = data.x[0].argmax().item()
        print(f"   • Node 0 is Protein Index: {active_protein_idx}")

# NODE EMBEDDING VAE 1

In [ ]:
import torch
import glob
import random
import numpy as np
from torch_geometric.nn import VGAE, GCNConv
import os

# --- CONFIGURATION ---
INPUT_DIM = 159       # Number of protein types
HIDDEN_DIM = 64       # Intermediate layer size
LATENT_DIM = 16       # Final embedding size
LEARNING_RATE = 0.01
EPOCHS = 10
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {DEVICE}")

# --- 1. THE MODEL ARCHITECTURE ---
class VariationalEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        # GCNConv aggregates information from neighbors
        self.conv1 = GCNConv(in_channels, hidden_channels)
        
        # Variational layers (Mean and Variance)
        self.conv_mu = GCNConv(hidden_channels, out_channels)
        self.conv_logstd = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        # Layer 1: Aggregation + ReLU activation
        x = self.conv1(x, edge_index).relu()
        
        # Layer 2: Output the parameters for the latent distribution
        return self.conv_mu(x, edge_index), self.conv_logstd(x, edge_index)

# Initialize the full VGAE model
encoder = VariationalEncoder(INPUT_DIM, HIDDEN_DIM, LATENT_DIM)
model = VGAE(encoder).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# --- 2. THE TRAINING LOOP ---
# Get list of files
files = glob.glob(f"{output_dir}/*.pt")
random.shuffle(files) # Shuffle initially

print(f"Starting training on {len(files)} graphs...")

for epoch in range(EPOCHS):
    total_loss = 0
    random.shuffle(files) # Shuffle every epoch to prevent cycle bias
    
    # Iterate through files one by one (Lazy Loading)
    for i, file_path in enumerate(files):
        # A. Load Data
        # weights_only=False suppresses the warning (safe since you made the files)
        data = torch.load(file_path, weights_only=False)
        data = data.to(DEVICE)
        
        # B. Forward Pass
        optimizer.zero_grad()
        
        # Encode: Get the latent 'z'
        z = model.encode(data.x, data.edge_index)
        
        # C. Calculate Loss
        # 1. Reconstruction Loss: Can the model predict the real edges from 'z'?
        recon_loss = model.recon_loss(z, data.edge_index)
        
        # 2. KL Divergence: Keep the latent space normal (Standard VAE regularization)
        # We normalize by num_nodes to stop KL from overpowering reconstruction on huge graphs
        kl_loss = (1 / data.num_nodes) * model.kl_loss()
        
        loss = recon_loss + kl_loss
        
        # D. Backprop
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Optional: Print progress every 50 files
        if i % 50 == 0:
            print(f"   Epoch {epoch+1} | File {i}/{len(files)} | Loss: {loss.item():.4f}")

    # End of Epoch Report
    avg_loss = total_loss / len(files)
    print(f"✅ Epoch {epoch+1} Completed. Avg Loss: {avg_loss:.4f}")

print("Training finished.")

# NODE EMBEDDING VAE 2

In [ ]:
import torch
import torch.nn as nn
import glob
import random
import numpy as np
from torch_geometric.nn import VGAE, GATv2Conv
import os

# --- CONFIGURATION ---
INPUT_DIM = 159       # Number of protein types
HIDDEN_DIM = 64       # Intermediate layer size
LATENT_DIM = 16       # Final embedding size
LEARNING_RATE = 0.005 # GAT usually prefers slightly lower LR than GCN
EPOCHS = 10
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Directories (Adjust if needed, assuming variables exist from previous context)
# output_dir = "..." 

print(f"Using device: {DEVICE}")

# --- 1. THE UPGRADED ATTENTION ARCHITECTURE ---
class VariationalGATEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        
        # Layer 1: Multi-Head Attention
        # heads=4 means we learn 4 different "views" of the neighborhood
        # We output (hidden_channels * heads) features
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=4, concat=True)
        
        # Calculate size after concatenation
        # Input to next layer will be hidden_channels * 4
        hidden_out = hidden_channels * 4
        
        # Variational Layers (Mean and Variance)
        # We use heads=1 and concat=False here to force a single, clean latent vector
        self.conv_mu = GATv2Conv(hidden_out, out_channels, heads=1, concat=False)
        self.conv_logstd = GATv2Conv(hidden_out, out_channels, heads=1, concat=False)

    def forward(self, x, edge_index):
        # Layer 1: Attention Aggregation + ELU activation (standard for GAT)
        x = self.conv1(x, edge_index).relu()
        
        # Layer 2: Predict Latent Distribution Parameters
        return self.conv_mu(x, edge_index), self.conv_logstd(x, edge_index)

# Initialize the Attention-based VGAE
encoder = VariationalGATEncoder(INPUT_DIM, HIDDEN_DIM, LATENT_DIM)
model = VGAE(encoder).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("🚀 Initialized GATv2 Spatial Encoder")

# --- 2. THE TRAINING LOOP ---
# Get list of files
files = glob.glob(f"{output_dir}/*.pt")
random.shuffle(files)

print(f"Starting training on {len(files)} graphs...")

for epoch in range(EPOCHS):
    total_loss = 0
    random.shuffle(files) 
    
    # Iterate through files
    for i, file_path in enumerate(files):
        # A. Load Data
        data = torch.load(file_path, weights_only=False)
        data = data.to(DEVICE)
        
        # B. Forward Pass
        optimizer.zero_grad()
        
        # Encode: Get the latent 'z' (Now spatially aware via Attention)
        z = model.encode(data.x, data.edge_index)
        
        # C. Calculate Loss
        # 1. Reconstruction Loss (Link Prediction)
        recon_loss = model.recon_loss(z, data.edge_index)
        
        # 2. KL Divergence
        # Normalized by num_nodes to keep scale consistent across small/large cells
        kl_loss = (1 / data.num_nodes) * model.kl_loss()
        
        loss = recon_loss + kl_loss
        
        # D. Backprop
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        if i % 50 == 0:
            print(f"   Epoch {epoch+1} | File {i}/{len(files)} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(files)
    print(f"✅ Epoch {epoch+1} Completed. Avg Loss: {avg_loss:.4f}")

    # Optional: Save checkpoint every epoch
    # torch.save(model.state_dict(), f"gvae_gat_epoch_{epoch}.pth")

print("Training finished.")
save_path = '/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/GVAE/models/gvae_gat_model.pth'

# Save the weights
torch.save(model.state_dict(), save_path)

print(f"💾 Model saved successfully to: {save_path}")

In [ ]:
torch.save(model.state_dict(), "/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/GVAE/models/vgae_model_epoch6.pth")
print("✅ Model saved safely.")

# MODEL LOAD

In [ ]:
load_attention_weights = True


if load_attention_weights:
    import torch
    import torch.nn as nn
    from torch_geometric.nn import VGAE, GATv2Conv
    import os

    # --- 1. CONFIGURATION (Must match training exactly) ---
    INPUT_DIM = 159       
    HIDDEN_DIM = 64       
    LATENT_DIM = 16       
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Path to your saved weights
    save_path = "/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/GVAE/models/gvae_gat_model.pth"

    # --- 2. DEFINE THE ARCHITECTURE (The Fix) ---
    # You must define this class exactly as it was during training
    class VariationalGATEncoder(torch.nn.Module):
        def __init__(self, in_channels, hidden_channels, out_channels):
            super().__init__()
            # Heads=4, concat=True -> Output size is hidden_channels * 4
            # This explains why the bias in your file is 256 (64 * 4)
            self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=4, concat=True)
            
            # Calculate input size for next layer
            hidden_out = hidden_channels * 4
            
            # Heads=1, concat=False -> Output size is out_channels
            self.conv_mu = GATv2Conv(hidden_out, out_channels, heads=1, concat=False)
            self.conv_logstd = GATv2Conv(hidden_out, out_channels, heads=1, concat=False)

        def forward(self, x, edge_index):
            x = self.conv1(x, edge_index).relu()
            return self.conv_mu(x, edge_index), self.conv_logstd(x, edge_index)

    # --- 3. LOAD THE MODEL ---
    print("🔄 Loading GATv2 model...")

    # A. Initialize the correct class
    encoder = VariationalGATEncoder(INPUT_DIM, HIDDEN_DIM, LATENT_DIM)
    model = VGAE(encoder)

    # B. Load the weights
    state_dict = torch.load(save_path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(state_dict)

    # C. Set to Evaluation Mode
    model.to(DEVICE)
    model.eval()

    print("✅ Model loaded successfully.")
    
else:
    # --- 1. CONFIGURATION (Must match training exactly) ---
    INPUT_DIM = 159       # Protein types
    HIDDEN_DIM = 64
    LATENT_DIM = 16
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # --- 2. DEFINE ARCHITECTURE ---
    class VariationalEncoder(torch.nn.Module):
        def __init__(self, in_channels, hidden_channels, out_channels):
            super().__init__()
            self.conv1 = GCNConv(in_channels, hidden_channels)
            self.conv_mu = GCNConv(hidden_channels, out_channels)
            self.conv_logstd = GCNConv(hidden_channels, out_channels)

        def forward(self, x, edge_index):
            x = self.conv1(x, edge_index).relu()
            return self.conv_mu(x, edge_index), self.conv_logstd(x, edge_index)

    # --- 3. INITIALIZE AND LOAD ---
    # Initialize the empty model
    encoder = VariationalEncoder(INPUT_DIM, HIDDEN_DIM, LATENT_DIM)
    model = VGAE(encoder).to(DEVICE)

    # Load the weights
    try:
        # Adjust filename if you named it something else
        model.load_state_dict(torch.load("/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/GVAE/models/vgae_model_epoch6.pth"))
        model.eval() # CRITICAL: Switch to evaluation mode
        print("✅ Model loaded successfully and set to eval mode.")
    except FileNotFoundError:
        print("❌ Error: Could not find 'vgae_model_epoch6.pth'. Check your file path.")

# MODEL EVAL

In [ ]:
model

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
import torch
import random
from torch_geometric.utils import negative_sampling

# 1. Pick a random file
test_file = random.choice(files)
data = torch.load(test_file, weights_only=False).to(DEVICE)
model.eval()

# 2. Get Latent Z
with torch.no_grad():
    z = model.encode(data.x, data.edge_index)

# 3. Create Test Set (Positive vs Negative Edges)
# Positive: The real edges in the graph
pos_edge_index = data.edge_index

# Negative: Random pairs that do NOT have an edge
# We ask for the same number of negative samples as positive edges
neg_edge_index = negative_sampling(
    edge_index=data.edge_index, 
    num_nodes=data.num_nodes,
    num_neg_samples=pos_edge_index.size(1)
)

# 4. Decode (Predict Probabilities)
# The decoder effectively does: Sigmoid( Z[u] dot Z[v] )
with torch.no_grad():
    pos_probs = model.decode(z, pos_edge_index, sigmoid=True)
    neg_probs = model.decode(z, neg_edge_index, sigmoid=True)

# 5. Prepare for Plotting
y_true = [1] * len(pos_probs) + [0] * len(neg_probs)
y_scores = torch.cat([pos_probs, neg_probs]).cpu().numpy()

# Calculate AUC
fpr, tpr, thresholds = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)

# --- PLOT 1: ROC Curve ---
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title(f'Link Prediction (Reconstruction)\nCell: {os.path.basename(test_file)}')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)

# --- PLOT 2: Probability Histograms ---
plt.subplot(1, 2, 2)
plt.hist(pos_probs.cpu().numpy(), bins=50, alpha=0.5, color='green', label='Real Edges')
plt.hist(neg_probs.cpu().numpy(), bins=50, alpha=0.5, color='red', label='Fake Edges')
plt.title('Prediction Confidence')
plt.xlabel('Predicted Probability')
plt.ylabel('Count')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Flatten z to see the distribution of all values across all dimensions
z_values = z.cpu().numpy().flatten()

plt.figure(figsize=(8, 4))
plt.hist(z_values, bins=100, color='purple', alpha=0.7, density=True)
plt.title('Distribution of Latent Values (z)')
plt.xlabel('Latent Value')
plt.ylabel('Density')
plt.axvline(0, color='black', linestyle='--', alpha=0.5)
plt.text(plt.xlim()[1]*0.7, plt.ylim()[1]*0.8, f"Mean: {z_values.mean():.2f}\nStd: {z_values.std():.2f}")
plt.grid(alpha=0.3)
plt.show()

# EXPLORE MARKERS

In [ ]:
import torch
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import os
from tqdm import tqdm

# --- CONFIGURATION ---
TARGET_MARKER = "CD69"           # The protein to extract
TARGET_CELL_GROUP = "CD4"       # Only process cells belonging to this group
MAX_MOLECULES = 100             # Downsample limit per cell

molecule_vectors = []
origin_cell_ids = []

# Filter the file list to include only cells that are in the "CD8" group
# This avoids loading unnecessary files for other cell types
valid_cells = set(adata.obs[adata.obs["cell_group"] == TARGET_CELL_GROUP].index)
filtered_files = [f for f in files if os.path.basename(f).replace(".pt", "") in valid_cells]

print(f"Processing {len(filtered_files)} cells (Group: {TARGET_CELL_GROUP})...")
model.eval()

with torch.no_grad():
    for file_path in tqdm(filtered_files):
        # 1. Load Data
        data = torch.load(file_path, weights_only=False).to(DEVICE)
        cell_id = os.path.basename(file_path).replace(".pt", "")
        
        # 2. Get Embeddings
        z_all = model.encode(data.x, data.edge_index)
        
        # 3. Identify CD8 Nodes
        m_idx = MARKER_TO_IDX.get(TARGET_MARKER)
        if m_idx is None: continue
        
        # Boolean mask for CD8 nodes
        is_target = (data.x.argmax(dim=1) == m_idx)
        z_target = z_all[is_target]
        
        # Skip if no CD8 nodes found in this cell
        if z_target.shape[0] == 0:
            continue
            
        # 4. Downsample (Max 500)
        if z_target.shape[0] > MAX_MOLECULES:
            # Randomly select 500 indices
            perm = torch.randperm(z_target.shape[0])[:MAX_MOLECULES]
            z_target = z_target[perm]
            
        # 5. Collect
        molecule_vectors.append(z_target.cpu().numpy())
        origin_cell_ids.extend([cell_id] * z_target.shape[0])

# Stack into one big matrix
X_molecules = np.vstack(molecule_vectors)
print(f"Extraction complete. Total CD8 molecules: {X_molecules.shape[0]}")

In [ ]:
# 1. Create Molecule-Level AnnData
adata_mol = ad.AnnData(X=X_molecules)
adata_mol.obs['origin_cell'] = origin_cell_ids

# 2. Map Metadata
# Ensure these column names match your adata.obs exactly
metadata_cols = ['cell_type', 'condition'] 

print("Mapping metadata...")
for col in metadata_cols:
    if col in adata.obs.columns:
        # Create lookup dictionary: Cell ID -> Value
        mapper = adata.obs[col].to_dict()
        # Map values to every molecule based on its origin cell
        adata_mol.obs[col] = adata_mol.obs['origin_cell'].map(mapper)
    else:
        print(f"⚠️ Warning: Column '{col}' not found in adata.obs")

print(adata_mol)

In [ ]:
# 1. Neighbors & UMAP
print("Running UMAP on molecules...")
sc.pp.pca(adata_mol, n_comps=15)
sc.pp.neighbors(
    adata_mol,
    n_neighbors=30,
    use_rep="X_pca",
)
sc.tl.umap(adata_mol)
sc.tl.leiden(adata_mol, resolution=0.5)


In [ ]:
metadata_cols = ['cell_type',
                 'condition'] 

print("Mapping metadata...")
for col in metadata_cols:
    if col in adata.obs.columns:
        # Create lookup dictionary: Cell ID -> Value
        mapper = adata.obs[col].to_dict()
        # Map values to every molecule based on its origin cell
        adata_mol.obs[col] = adata_mol.obs['origin_cell'].map(mapper)
    else:
        print(f"⚠️ Warning: Column '{col}' not found in adata.obs")

print(adata_mol)

In [ ]:
sc.set_figure_params(figsize=(10, 18), dpi=150)

sc.pl.umap(
    adata_mol, 
    color=[ 'cell_type', 'condition'],
    title=['Spatial Context Clusters', 'cell type',  'Condition'],
    wspace=0.35,
    ncols=2,
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

obs = adata_mol.obs.copy()

# ensure categorical
for c in ["leiden", "condition", "cell_type"]:
    obs[c] = obs[c].astype("category")

# ---- cluster sizes ----
cluster_sizes = obs["leiden"].value_counts().sort_values(ascending=True)
leiden_order = cluster_sizes.index

# ---- proportions ----
def prop_table(row_key, col_key, order):
    return (
        obs.groupby([row_key, col_key])
        .size()
        .groupby(level=0)
        .apply(lambda x: x / x.sum())
        .unstack(fill_value=0)
        .loc[order]
    )

cond_prop = prop_table("leiden", "condition", leiden_order)
cta_prop  = prop_table("leiden", "cell_type", leiden_order)

# ---- pretty y-axis labels with cluster size ----
ylabels = [f"{l} (n={cluster_sizes[l]})" for l in leiden_order]

# ---- plotting ----
fig, axes = plt.subplots(
    1, 2,
    figsize=(16, max(6, 0.4 * len(cond_prop))),
    sharey=True
)

# --- condition plot ---
cond_prop.plot(
    kind="barh",
    stacked=True,
    ax=axes[0],
    width=0.9,
    legend=True
)
axes[0].set_title("Condition composition per Leiden (sorted by size)")
axes[0].set_xlabel("Proportion")
axes[0].set_ylabel("Leiden cluster")
axes[0].set_yticklabels(ylabels)
axes[0].invert_yaxis()  # largest cluster on top

# --- cell type plot ---
cta_prop.plot(
    kind="barh",
    stacked=True,
    ax=axes[1],
    width=0.9,
    legend=True
)
axes[1].set_title("Cell-type abundance per Leiden (sorted by size)")
axes[1].set_xlabel("Proportion")
axes[1].legend(title="Cell type", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()


# CREATE NODE HISTOGRAM FEATURES

In [ ]:
import torch
import numpy as np
import pandas as pd
from sklearn.cluster import MiniBatchKMeans
from torch_geometric.loader import DataLoader
from torch.utils.data import Dataset
from tqdm import tqdm
import os
import joblib

# --- 1. CONFIGURATION ---
ALL_MARKERS = list(MARKER_TO_IDX.keys())

WORDS_PER_MARKER = 15       # 10 spatial patterns per protein
N_FILES_TO_SAMPLE = 500     # Look at 500 random cells for vocabulary learning
MAX_NODES_PER_FILE = 1000   # Cap contributions to 1,000 nodes per cell (Diversity)

BATCH_SIZE = 16
NUM_WORKERS = 4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- DIRECTORY SETUP ---
# Update these paths if you want to save elsewhere
BASE_PATH = "/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/GVAE/attention_nodes"
MODEL_DIR = os.path.join(BASE_PATH, "vocabs")
CACHE_DIR = os.path.join(BASE_PATH, "features")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"🚀 Starting Resumable Pipeline for {len(ALL_MARKERS)} proteins.")
print(f"   • Sampling Strategy: Up to {MAX_NODES_PER_FILE} nodes from {N_FILES_TO_SAMPLE} random cells.")
print(f"   • Models: {MODEL_DIR}")
print(f"   • Data Cache: {CACHE_DIR}")

# --- 2. DEFINE DATA LOADER ---
class GraphDataset(Dataset):
    def __init__(self, file_list):
        self.file_list = file_list
    def __len__(self):
        return len(self.file_list)
    def __getitem__(self, idx):
        return torch.load(self.file_list[idx], weights_only=False)

dataset = GraphDataset(files)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# --- 3. MAIN RESUMABLE LOOP ---
for i, marker in enumerate(ALL_MARKERS):
    
    # A. Check Cache (Skip if done)
    feature_save_path = os.path.join(CACHE_DIR, f"{marker}_features.pkl")
    
    if os.path.exists(feature_save_path):
        print(f"[{i+1}/{len(ALL_MARKERS)}] {marker}: Found in cache. Skipping.")
        continue

    print(f"\n[{i+1}/{len(ALL_MARKERS)}] {marker}: Processing...")
    
    if marker not in MARKER_TO_IDX:
        print(f"   ⚠️ Warning: {marker} not in dictionary. Skipping.")
        continue

    # --- PHASE A: LEARN VOCABULARY (Broad Sampling) ---
    sample_embeddings = []
    
    try:
        model.eval()
        with torch.no_grad():
            # 1. Select random subset of files
            n_select = min(N_FILES_TO_SAMPLE, len(files))
            shuffled_files = np.random.choice(files, size=n_select, replace=False)
            
            for f in shuffled_files:
                try:
                    data = torch.load(f, weights_only=False).to(DEVICE)
                    
                    # Run Inference
                    z_all = model.encode(data.x, data.edge_index)
                    
                    # Filter for target protein
                    m_idx = MARKER_TO_IDX[marker]
                    is_target = (data.x.argmax(dim=1) == m_idx)
                    z_target = z_all[is_target].cpu().numpy()
                    
                    count = z_target.shape[0]
                    if count > 0:
                        # 2. Subsample if too dense (The "Diversity Cap")
                        if count > MAX_NODES_PER_FILE:
                            idx_keep = np.random.choice(count, size=MAX_NODES_PER_FILE, replace=False)
                            z_target = z_target[idx_keep]
                        
                        sample_embeddings.append(z_target)
                        
                except Exception as e:
                    # Individual file errors (rare) shouldn't stop the loop
                    continue

        # 3. Check Sampling Yield
        if not sample_embeddings:
            print(f"   ⚠️ Skipping {marker}: No nodes found in {n_select} sampled cells.")
            # Save empty placeholder so we don't retry endlessly
            pd.DataFrame().to_pickle(feature_save_path)
            continue

        X_sample = np.vstack(sample_embeddings)
        total_samples = X_sample.shape[0]
        
        # Safety Check: Do we have enough data for 10 clusters?
        if total_samples < WORDS_PER_MARKER * 10:
             print(f"   ⚠️ Skipping {marker}: Not enough data ({total_samples} nodes).")
             pd.DataFrame().to_pickle(feature_save_path)
             continue

        # 4. Train K-Means
        print(f"   • Learning vocabulary from {total_samples} nodes (from {n_select} cells)...")
        kmeans = MiniBatchKMeans(n_clusters=WORDS_PER_MARKER, batch_size=4096, random_state=42, n_init='auto')
        kmeans.fit(X_sample)
        
        # Save Model
        joblib.dump(kmeans, os.path.join(MODEL_DIR, f"kmeans_{marker}.pkl"))
        
        # --- PHASE B: ENCODE ALL CELLS ---
        print(f"   • Encoding all cells...")
        marker_histograms = {} 
        
        with torch.no_grad():
            for batch_idx, batch in enumerate(loader): 
                batch = batch.to(DEVICE)
                z_all = model.encode(batch.x, batch.edge_index).cpu().numpy()
                
                batch_x_idx = batch.x.argmax(dim=1).cpu().numpy()
                is_target = (batch_x_idx == MARKER_TO_IDX[marker])
                
                batch_ids = batch.batch.cpu().numpy()
                unique_cells = np.unique(batch_ids)
                
                for b_id in unique_cells:
                    # Calculate File Index
                    file_idx = (batch_idx * BATCH_SIZE) + b_id
                    if file_idx >= len(dataset.file_list): continue
                        
                    filename = dataset.file_list[file_idx]
                    cell_name = os.path.basename(filename).replace(".pt", "")
                    
                    mask = (batch_ids == b_id) & is_target
                    
                    if not np.any(mask):
                        hist = np.zeros(WORDS_PER_MARKER)
                    else:
                        z_cell = z_all[mask]
                        words = kmeans.predict(z_cell)
                        # Density=True ensures sums to 1 (Frequency)
                        hist, _ = np.histogram(words, bins=range(WORDS_PER_MARKER + 1), density=True)
                    
                    marker_histograms[cell_name] = hist

        # --- SAVE RESULT TO DISK ---
        df_marker = pd.DataFrame.from_dict(marker_histograms, orient='index')
        df_marker.columns = [f"{marker}_W{w}" for w in range(WORDS_PER_MARKER)]
        
        df_marker.to_pickle(feature_save_path)
        print(f"   ✅ Saved {marker} features.")

    except Exception as e:
        print(f"   ❌ CRITICAL ERROR on {marker}: {e}")
        # We continue to the next marker so one failure doesn't kill the run
        continue

# --- 4. MERGE (Run this when everything finishes) ---
print("\n--- Merging Cached Features ---")
all_feature_dfs = []

for marker in ALL_MARKERS:
    feature_save_path = os.path.join(CACHE_DIR, f"{marker}_features.pkl")
    if os.path.exists(feature_save_path):
        try:
            df = pd.read_pickle(feature_save_path)
            if not df.empty:
                all_feature_dfs.append(df)
        except Exception:
            pass

if all_feature_dfs:
    df_spatial = pd.concat(all_feature_dfs, axis=1).fillna(0)
    df_spatial = df_spatial.sort_index()
    print(f"✅ DONE! Final Massive Spatial Atlas: {df_spatial.shape}")
else:
    print("⚠️ No valid data found in cache.")

In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd
import os

# --- CONFIGURATION ---
SAVE_PATH = "/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/GVAE/spatial_atlas_ALL_PROTEINS.h5ad"
ATTENTION_SAVE_PATH = "/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/GVAE/attention_nodes/spatial_atlas_ALL_PROTEINS_ATTENTION.h5ad"

print("📦 Packaging Massive Spatial Atlas (Speed Mode)...")

# 1. ALIGNMENT
# Align new features to match the exact cell order of the original metadata
print("   • Aligning spatial features to original metadata...")
df_spatial_aligned = df_spatial.reindex(adata.obs_names)

# Handle missing cells (fill with 0)
if df_spatial_aligned.isna().any().any():
    print("     ⚠️ Warning: Some cells were missing spatial data. Filling with zeros.")
    df_spatial_aligned = df_spatial_aligned.fillna(0)

# 2. Create AnnData
adata_spatial = ad.AnnData(df_spatial_aligned)

# 3. Transfer Metadata
adata_spatial.obs = adata.obs.copy()

# 4. Add Annotations
adata_spatial.uns['spatial_markers'] = list(MARKER_TO_IDX.keys())
adata_spatial.uns['spatial_granularity'] = WORDS_PER_MARKER 

# 5. Save to Disk (FAST MODE)
# Removing 'compression' makes writing/reading much faster, but file size will be larger.
adata_spatial.write(ATTENTION_SAVE_PATH)

print(f"\n✅ SUCCESS: Saved Spatial Atlas to '{ATTENTION_SAVE_PATH}'")
print(f"   • Shape: {adata_spatial.shape}")
print(f"   • Features: {adata_spatial.n_vars} spatial words")

# --- VERIFICATION ---
print("\n🔍 Verifying file integrity...")
test_load = sc.read_h5ad(ATTENTION_SAVE_PATH)
print(f"   • Reloaded successfully! Ready for full-scale VAE training.")

# VAE MODEL

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

# --- CONFIGURATION ---
BATCH_SIZE = 64
EPOCHS = 300
LATENT_DIM = 32
LEARNING_RATE = 1e-3
INPUT_DIM = adata_spatial.n_vars # likely 2385
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- 1. PREPARE DATA ---
# Ensure data is strictly 0-1 for BCE Loss
print("Preparing Data...")
X_numpy = adata_spatial.X.copy()
if not isinstance(X_numpy, np.ndarray):
    X_numpy = X_numpy.toarray()

# Clip to ensure numerical stability for BCE (0.0 to 1.0)
X_numpy = np.clip(X_numpy, 0, 1).astype(np.float32)

X_train, X_test = train_test_split(X_numpy, test_size=0.1, random_state=42)

train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train)), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TensorDataset(torch.from_numpy(X_test)), batch_size=BATCH_SIZE, shuffle=False)

# --- 2. THE MODEL (With BatchNorm for Stability) ---
class RobustCellVAE(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super(RobustCellVAE, self).__init__()
        
        # Encoder: Deeper and wider to handle 2385 features
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.2),
            
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.2)
        )
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        
        # Decoder: Mirror of Encoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.2),
            
            nn.Linear(128, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2),
            
            nn.Linear(512, input_dim),
            nn.Sigmoid() # Force output to 0-1 (frequencies)
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        hidden = self.encoder(x)
        mu = self.fc_mu(hidden)
        logvar = self.fc_logvar(hidden)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar

# --- 3. LOSS FUNCTION WITH ANNEALING ---
def vae_loss(recon_x, x, mu, logvar, beta):
    """
    beta: The weight of the KL term. We change this over time.
    """
    # 1. Reconstruction (BCE is better for sparse frequencies than MSE)
    # sum across features, then average across batch
    BCE = nn.functional.binary_cross_entropy(recon_x, x, reduction='sum') / x.size(0)
    
    # 2. KL Divergence
    # sum across latent dims, then average across batch
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    
    return BCE + beta * KLD, BCE, KLD

# --- 4. TRAINING LOOP ---
model = RobustCellVAE(INPUT_DIM, LATENT_DIM).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"🚀 Training Robust VAE with KL Annealing...")
history = {'loss': [], 'bce': [], 'kld': []}

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    train_bce = 0
    train_kld = 0
    
    # KL ANNEALING STRATEGY
    # Ramp up Beta from 0.0 to 0.05 over the first 20 epochs
    # This lets the model learn Features first, structure second.
    beta = min(0.05, (epoch / 20) * 0.05) 
    
    for batch in train_loader:
        x_batch = batch[0].to(DEVICE)
        optimizer.zero_grad()
        
        recon, mu, logvar = model(x_batch)
        loss, bce, kld = vae_loss(recon, x_batch, mu, logvar, beta)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        train_bce += bce.item()
        train_kld += kld.item()
    
    # Logging
    avg_loss = train_loss / len(train_loader)
    avg_bce = train_bce / len(train_loader)
    avg_kld = train_kld / len(train_loader)
    
    history['loss'].append(avg_loss)
    history['bce'].append(avg_bce)
    history['kld'].append(avg_kld)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:02d} | Loss: {avg_loss:.2f} (Rec: {avg_bce:.2f} | KL: {avg_kld:.2f} | β: {beta:.4f})")

# --- 5. DIAGNOSTIC PLOT ---


plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.plot(history['loss'], label='Total Loss')
plt.title("Total Loss")
plt.subplot(1, 3, 2)
plt.plot(history['bce'], color='orange', label='Reconstruction')
plt.title("Reconstruction (BCE)")
plt.subplot(1, 3, 3)
plt.plot(history['kld'], color='green', label='KL Divergence')
plt.title("Latent Structure (KLD)")
plt.show()

# --- 6. SAVE EMBEDDINGS ---
model.eval()
with torch.no_grad():
    full_tensor = torch.from_numpy(X_numpy).to(DEVICE)
    _, mu, _ = model(full_tensor)
    adata_spatial.obsm['X_spatial_vae'] = mu.cpu().numpy()

print(f"✅ Saved robust embeddings to adata_spatial.obsm['X_spatial_vae']")

In [ ]:
model

In [ ]:
# 1. Save the embeddings to the AnnData object
model.eval()
with torch.no_grad():
    # Get the latent means (mu) for all cells
    full_tensor = torch.from_numpy(X_numpy).to(DEVICE)
    _, mu, _ = model(full_tensor)
    adata_spatial.obsm['X_spatial_vae'] = mu.cpu().numpy()

print(f"✅ Embeddings saved. Shape: {adata_spatial.obsm['X_spatial_vae'].shape}")

# 2. Run UMAP on these new embeddings
print("running UMAP...")
sc.pp.neighbors(adata_spatial, use_rep='X_spatial_vae', n_neighbors=30)
sc.tl.umap(adata_spatial)

# 3. Plot
# If the model worked, you should see clear clusters or gradients
sc.pl.umap(
    adata_spatial, 
    color=['condition', 'cell_type'], 
    title=['Deep Spatial State (Condition)', 'Deep Spatial State (Cell Type)'],
    wspace=0.3
)

In [ ]:
adata_spatial.obsm['X_spatial_vae'].shape

In [ ]:
np.save("/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/GVAE/final_embeddings.npy", adata_spatial.obsm['X_spatial_vae'])
